In [17]:
import os
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import timm
from PIL import Image
%load_ext autoreload
%autoreload 2

# hyperparameters
device = torch.device("cpu")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
# initialize model
model = timm.create_model("hf-hub:BVRA/MegaDescriptor-T-224", pretrained=True, num_classes=0)

# load state dict containing miscellaneous state or just the model weights
state_dict = torch.load("../weights/MegaD_elephant_w.pt", map_location=device)

if "optimizer" in state_dict:
    model.load_state_dict(state_dict["model"])
else:
    model.load_state_dict(state_dict)

model.to(device)
model.eval()

/tmp/ipykernel_3635223/2966897652.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load("../weights/MegaD_elephant_w.pt", map_location=device)


SwinTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (layers): Sequential(
    (0): SwinTransformerStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): SwinTransformerBlock(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): WindowAttention(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (softmax): Softmax(dim=-1)
          )
          (drop_path1): Identity()
          (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (act): GELU(approximate='none')
            (drop1): 

In [19]:
def compute_embeddings(image):
    with torch.inference_mode():
        
        image = image.unsqueeze(0).to(device)  
        embedding = model(image)
    return embedding

In [20]:
from data_handler import EleHandler
import os 
from pathlib import Path

elephant = EleHandler(dataset_dir=Path('/archive/vision/beery/animal_reid/datasets/elephants_zooniverse'),
                       dictonary_path=Path('/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary.csv'),
                       transform='MegaDescriptor-elephant'
                       )
#elephant.print_image(0)
print(elephant[0])
image, subject_id, ele_id, identified, subject_SEEK, ele_SEEK = elephant[2]
print('subject_id:', subject_id)
print('ele_id:', ele_id)
print('identified:', identified)
print('subject_SEEK:', subject_SEEK)
print('ele_SEEK:', ele_SEEK)
#image.show()

(tensor([[[ 0.6906,  0.5707,  0.4337,  ..., -0.5253, -0.1486,  0.1254],
         [ 0.6563,  0.6221,  0.5536,  ..., -0.4739, -0.1143,  0.1426],
         [ 0.7077,  0.7419,  0.6906,  ..., -0.3198,  0.0056,  0.0398],
         ...,
         [ 0.7933,  0.8961,  1.1187,  ...,  0.9646,  1.0159,  1.0159],
         [ 0.9132,  1.0331,  1.1358,  ...,  1.0331,  1.1187,  1.0673],
         [ 1.0159,  1.0673,  1.1015,  ...,  1.0673,  1.1187,  1.1358]],

        [[ 0.6254,  0.5903,  0.5378,  ..., -0.3901,  0.0301,  0.3102],
         [ 0.6429,  0.6429,  0.5728,  ..., -0.3375,  0.0651,  0.3452],
         [ 0.7304,  0.7304,  0.6779,  ..., -0.2150,  0.1352,  0.2577],
         ...,
         [ 0.7479,  0.8880,  1.0805,  ...,  0.9230,  0.9230,  0.9230],
         [ 0.8529,  0.9930,  1.0980,  ...,  0.9755,  1.0105,  0.9580],
         [ 0.9580,  1.0280,  1.0280,  ...,  0.9580,  1.0105,  1.0280]],

        [[ 0.2522,  0.2348,  0.1651,  ..., -0.4450, -0.1138,  0.0779],
         [ 0.3045,  0.3045,  0.1999,  ..., -

In [21]:
embedding = compute_embeddings(image)
print('embedding shape:', embedding.shape)
print(embedding)

embedding shape: torch.Size([1, 768])
tensor([[ 8.8947e-01, -2.2840e-01,  5.0363e-01,  6.0970e-01, -4.6782e-01,
         -9.3296e-01, -7.9007e-01,  4.5128e-01,  9.4851e-01, -4.7791e-01,
         -1.9890e-01,  7.7828e-01, -9.6566e-01, -5.3021e-01,  3.7662e-01,
         -7.0048e-01, -1.7122e-01,  8.2352e-02, -2.5987e-01,  1.2285e-01,
         -4.3948e-01, -5.5433e-01, -6.4731e-01, -1.0377e+00,  2.5358e-01,
          1.3486e-01, -9.9599e-01,  6.2393e-01,  8.0866e-01, -5.0130e-02,
         -1.9514e-01,  6.3885e-01, -6.0360e-01,  1.7337e-01,  3.3681e-01,
          5.8922e-01, -8.1372e-02,  1.0448e+00, -1.7265e-01,  2.4322e-01,
         -5.2675e-01, -8.3415e-03, -1.3205e+00,  4.6089e-01,  1.1224e-01,
          1.4254e+00,  2.2168e-01,  3.5109e-01, -7.0306e-01,  1.6277e-01,
          6.0804e-02, -1.0072e-02, -1.5561e-01, -5.1135e-01, -1.6185e-01,
         -3.0817e-01, -5.7239e-01,  2.6307e-01,  1.3882e+00, -8.6101e-01,
         -8.9803e-01,  7.0693e-02,  1.3852e-01, -1.1800e+00, -5.4751e-01,


In [ ]:
# Make a dataloader that mixes wevrything randomly loads the images splits a train and test and computes the embeddings
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler

# Create dataset and dataloaders
dataset = EleHandler(dataset_dir=Path('/archive/vision/beery/animal_reid/datasets/elephants_zooniverse'),
                       dictonary_path=Path('/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary.csv'),
                       transform='MegaDescriptor-elephant'
                       )


train_size = int(0.7 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=128, sampler=RandomSampler(train_dataset))
val_loader = DataLoader(val_dataset, batch_size=128, sampler=RandomSampler(val_dataset))
test_loader = DataLoader(test_dataset, batch_size=128, sampler=RandomSampler(test_dataset))

# test

In [33]:
#layer = torch.nn.Linear(768, num_concepts).to(device)

def epoch_pass(epoch, loader, training = True):
    #layer.train() if training else layer.eval()
    for param in model.parameters(): # Freezing the MegaDescriptor
        param.requires_grad = False

    #optimizer = torch.optim.Adam(layer.parameters(), lr=0.001)
    

    for batch in loader:
        images, subject_id, ele_id, identified, subject_SEEK, ele_SEEK = batch
        print(len(subject_SEEK))
        subject_SEEK_tensor = torch.tensor(subject_SEEK)
        print('subject_SEEK shape:', subject_SEEK_tensor.shape)
        
        embeddings = compute_embeddings(images)
        #outputs = layer(embeddings)
        
        #loss = F.cross_entropy(outputs, subject_SEEK)
        
        #if training:
        #    optimizer.zero_grad()
        #    loss.backward()
        #    optimizer.step()


In [34]:

for epoch in range(5):
    train_loss, train_acc = epoch_pass(epoch, train_loader, training = True)
    val_loss, val_acc = epoch_pass(epoch, val_loader, training = False)



128


ValueError: too many dimensions 'str'